In [52]:
# Cell 1: Basic imports
from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings


In [53]:
# Cell 2: MANUALLY LOAD A PDF FILE

from pathlib import Path
from pypdf import PdfReader

ROOT = Path.cwd().parent   # notebooks/ → parent = main project folder
PDF_PATH = ROOT / "datas" / "ipc.pdf"    # correct path

print("Looking for PDF at:", PDF_PATH)

def load_pdf_text(path):
    reader = PdfReader(str(path))
    pages = []
    for p in reader.pages:
        txt = p.extract_text()
        if txt:
            pages.append(txt)
    return " ".join(" ".join(pages).split())

raw_text = load_pdf_text(PDF_PATH)

print("Characters extracted:", len(raw_text))
print(raw_text[:500])


Looking for PDF at: d:\Legal Chatbot using RAG\datas\ipc.pdf
Characters extracted: 4475722
THE INDIAN PENAL CODE CHAPTER I INTRODUCTION The Indian Penal Code was drafted by the First Indian Law Commission presided over by Lord Thomas Babington Macaulay. The draft underwent further revision at the hands of well-known jurists, like Sir Barnes Peacock, and was completed in 1850. The Indian Penal Code was passed by the then Legislature on 6 October 1860 and was enacted as Act No. XLV of 1860. Preamble. WHEREAS it is expedient to provide a general Penal Code for India; It is enacted as fol


In [54]:
# Cell 3: MANUAL CHUNKING
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += (chunk_size - overlap)
    return chunks

chunks = chunk_text(raw_text, chunk_size=700, overlap=100)
print("Total chunks:", len(chunks))
print(chunks[0][:300])


Total chunks: 7460
THE INDIAN PENAL CODE CHAPTER I INTRODUCTION The Indian Penal Code was drafted by the First Indian Law Commission presided over by Lord Thomas Babington Macaulay. The draft underwent further revision at the hands of well-known jurists, like Sir Barnes Peacock, and was completed in 1850. The Indian P


In [55]:
# Cell 4: EMBEDDINGS (no API needed)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")  # small & fast

embeddings = embed_model.encode(chunks, show_progress_bar=True).tolist()

print("Embedding dimension:", len(embeddings[0]))


Batches:   0%|          | 0/234 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Safe: rename existing chroma_db -> chroma_db_backup (keeps a copy, no deletes)
from pathlib import Path
import os, sys

p = Path.cwd() / "chroma_db"
if not p.exists():
    print("No chroma_db folder found in", Path.cwd())
else:
    backup = Path.cwd() / "chroma_db_backup"
    # if backup exists, add suffix with timestamp
    if backup.exists():
        import datetime
        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        backup = Path.cwd() / f"chroma_db_backup_{ts}"
    p.rename(backup)
    print(f"Renamed existing folder:\n  {p}\n->\n  {backup}\nBackup kept. You can inspect it later.")

print("\nNow restart the notebook kernel (Kernel → Restart) and re-run the Chroma initialization cell.")
print("After restart, run your Cell 6 that creates the chroma client; it will create a fresh chroma_db folder automatically.")


No chroma_db folder found in d:\Legal Chatbot using RAG\notebooks

Now restart the notebook kernel (Kernel → Restart) and re-run the Chroma initialization cell.
After restart, run your Cell 6 that creates the chroma client; it will create a fresh chroma_db folder automatically.


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_core.documents import Document   # <-- correct import for latest LC

# Convert chunks into Document objects
docs = [Document(page_content=chunk, metadata={}) for chunk in chunks]

# Initialize embedding model
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Create or load vectorstore
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

# Save to disk
vectorstore.persist()   
print("Vectorstore created & saved.")


Vectorstore created & saved.


C:\Users\KARTHIK\AppData\Local\Temp\ipykernel_57848\3676606194.py:19: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def search(query, top_k=3):
    retriever.search_kwargs["k"] = top_k
    docs = retriever.invoke(query)
    return docs

query = "What does this law say about cheating or fraud?"
results = search(query, top_k=3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content[:500])



--- Result 1 ---
 which property is transferred a more speciﬁc provision is made by section 420. The offence of cheating is not committed if a third party, on whom no deception has been practised, sustains pecuniary loss in consequence of the accused's act.475. [s 415.2] Cheating and extortion.— The offence of cheating must, like that of extortion, be committed by the wrongful obtaining of a consent. The difference is that the extortioner obtains the consent by intimidation, and the cheat by deception.476. [s 41

--- Result 2 ---
e was hired, obtains pay to which he is not entitled. "In all these cases there is deception. In all, the deceiver's object is fraudulent. He intends in all these cases to acquire or retain wrongful possession of that to which some other person has a better claim and which that other person is entitled to recover by law. In all these cases, therefore, the object has been wrongful gain, attended with wrongful loss. In all, therefore, there has, according to ou

In [ ]:
from dotenv import load_dotenv
load_dotenv()   # Loads .env file automatically

import os
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

if DEEPSEEK_API_KEY is None:
    raise ValueError("DEEPSEEK_API_KEY not found. Check your .env file.")
else:
    print("DeepSeek API key loaded safely from .env")


DeepSeek API key loaded safely from .env


In [56]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name="deepseek-chat",
    openai_api_key=os.getenv("DEEPSEEK_API_KEY"),
    openai_api_base="https://api.deepseek.com/v1"
)


In [57]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableMap
from langchain_core.output_parsers import StrOutputParser
load_dotenv()
# Prompt template
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a legal assistant specializing in Indian Law.
Use ONLY the context below to answer clearly and accurately.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""
)

# --- RAG PIPELINE ---
def rag_answer(question, top_k=3):

    # Retrieve documents
    docs = search(question, top_k)
    context = "\n\n".join([d.page_content for d in docs])

    # Build a runnable chain
    chain = (
        RunnableMap({
            "context": lambda _: context,
            "question": lambda _: question
        })
        | prompt
        | llm        # DeepSeek ChatOpenAI
        | StrOutputParser()
    )

    return chain.invoke({})

# Test
print(rag_answer("Explain IPC cheating section.", 3))


AuthenticationError: Error code: 401 - {'error': {'message': 'Authentication Fails, Your api key: ****95df is invalid', 'type': 'authentication_error', 'param': None, 'code': 'invalid_request_error'}}

In [60]:
# Reset & reload env safely, then test DeepSeek key
import os
from dotenv import load_dotenv

# Remove any previously set in-session key (safe)
os.environ.pop("DEEPSEEK_API_KEY", None)

# Load .env and force override (so .env value wins)
load_dotenv(override=True)

key = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY present:", key is not None)
if key:
    print("key starts:", key[:5], "ends:", key[-5:], "len:", len(key))
else:
    print("No DEEPSEEK_API_KEY found. Ensure .env in project root with no quotes.")


DEEPSEEK_API_KEY present: True
key starts: sk-9d ends: 4c187 len: 35


In [61]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model_name="deepseek-chat",
    openai_api_key=os.getenv("DEEPSEEK_API_KEY"),
    openai_api_base="https://api.deepseek.com/v1"
)

# quick smoke test
print("LLM test ->", llm.invoke("Say hello in one sentence."))


APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient Balance', 'type': 'unknown_error', 'param': None, 'code': 'invalid_request_error'}}

In [63]:
# Robust HF RAG wrapper that handles different InferenceClient signatures
from dotenv import load_dotenv
load_dotenv(override=True)
import os, inspect
from huggingface_hub import InferenceClient

# load token (try multiple env var names)
HF_TOKEN = (
    os.getenv("HUGGINGFACEHUB_API_TOKEN")
    or os.getenv("HUGGING_FACE_API")
    or os.getenv("HUGGINGFACE_API")
    or os.getenv("HF_TOKEN")
    or os.getenv("HUGGINGFACEHUB_API")
)

if not HF_TOKEN:
    raise ValueError("Hugging Face token not found. Put your key in .env as HUGGINGFACEHUB_API_TOKEN or HF_TOKEN etc.")

print("HF token length:", len(HF_TOKEN))

client = InferenceClient(token=HF_TOKEN)
HF_MODEL = "google/flan-t5-large"   # change if you prefer another model

def _call_text_generation(client, model, prompt, **kwargs):
    """
    Try several common signatures for text_generation:
    - model=..., inputs=...
    - model=..., input=...
    - model=..., prompt=...
    - generate(...) or text_generation(...) variations
    Returns the raw response.
    """
    # preference order for kw name
    try_names = ["inputs", "input", "prompt", "prompt_text"]
    # try text_generation with different kwarg names
    for name in try_names:
        try:
            call_args = { "model": model, name: prompt }
            call_args.update(kwargs)
            fn = getattr(client, "text_generation", None)
            if callable(fn):
                return fn(**call_args)
        except TypeError as e:
            # signature mismatch — try next
            continue
        except Exception as e:
            # other error (rate limit, auth) — raise it
            raise

    # if text_generation isn't present or failed, try generic generate or inference endpoints
    # try 'generate' function
    try:
        fn2 = getattr(client, "generate", None)
        if callable(fn2):
            return fn2(model=model, inputs=prompt, **kwargs)
    except TypeError:
        try:
            return fn2(model=model, input=prompt, **kwargs)
        except Exception:
            pass
    # fallback: try client.post / client.request raw (less common)
    raise RuntimeError("Could not call text_generation: incompatible huggingface_hub client signature in this env.")

def _extract_generated_text(resp):
    # safe extractor for common response shapes
    try:
        if isinstance(resp, list) and len(resp) > 0 and isinstance(resp[0], dict):
            for key in ("generated_text", "text", "content", "result"):
                if key in resp[0]:
                    return resp[0][key]
            return str(resp[0])
        if isinstance(resp, dict):
            for key in ("generated_text", "text", "content", "result"):
                if key in resp:
                    return resp[key]
            # some clients return {"outputs": [{"generated_text": ...}]}
            if "outputs" in resp and isinstance(resp["outputs"], list) and resp["outputs"]:
                out0 = resp["outputs"][0]
                if isinstance(out0, dict) and "generated_text" in out0:
                    return out0["generated_text"]
            return str(resp)
        if hasattr(resp, "generated_text"):
            return getattr(resp, "generated_text")
    except Exception:
        pass
    return str(resp)

def rag_answer_hf(question, top_k=3, debug=False):
    docs = search(question, top_k)
    context = "\n\n".join([d.page_content for d in docs])

    prompt = f"""You are a legal assistant specializing in Indian law.
Use ONLY the context below to answer clearly and accurately. Cite section numbers if present.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    # try calling HF text generation with a few common names
    try:
        raw = _call_text_generation(client, HF_MODEL, prompt, max_new_tokens=256)
    except Exception as e:
        # show helpful message for auth/permission issues
        if "401" in str(e) or "Authentication" in str(e) or "invalid" in str(e).lower():
            raise
        raise RuntimeError(f"HuggingFace call failed: {e}")

    if debug:
        print("RAW RESPONSE:", raw)

    return _extract_generated_text(raw)

# quick test (set debug=True to inspect raw shape)
print(rag_answer_hf("Explain IPC cheating section.", top_k=3, debug=True)[:1200])


HF token length: 37


RuntimeError: HuggingFace call failed: 